# CS-781 Phase 7 - Prior Case Retrieval (PCR): all experiments in one notebook

**Setup on Kaggle:** Accelerator *GPU T4 x2*, Internet on, secret `HF` (your Hugging Face token), and the same four inputs as Phases 4-6:
`export_gqa`, `export_mla`, `export_retnet` (datasets) and the `Phase1-Arch3` notebook output (contains `export_encdec`).
The smoke check passed (4 Nov), so `SMOKE = False` is set: use *Save Version -> Save & Run All* (~4 h).

**1. Data preparation (CPU, `pcr_prep.py`, once).** Parses the three splits and their own candidate pools, prints the data analysis
(pool sizes, gold-set sizes, gold missing from the pool, the best F1@7 any system could reach, document lengths, overlaps), and builds a
from-scratch **BM25** whose settings (k1, b, unigrams vs uni+bigrams, query term weighting) are tuned on dev. It also checks two non-neural
signals on dev: a candidate that mentions a later year than any year in the query cannot be a *prior* case, and how often a candidate is cited by
training queries. For every training query it stores gold + BM25 top-60 (hard negatives, and the candidate list for distillation).

**2. Retriever (shared by every run, so the comparison is about the adaptation technique and the backbone).** A bi-encoder on the Phase-1 backbone:
each judgment is cut into 512-token chunks; 16 chunks evenly spaced over the whole judgment are stored, 8 are read at inference (4,096 tokens).
Every chunk is mean-pooled and projected to a 256-d unit vector. Relevance = **chunk max-sim** (every query chunk finds its best-matching candidate chunk,
averaged over query chunks); the single-vector cosine of mean chunk vectors is scored too (pooling comparison, free). Training: InfoNCE over the step
(sampled gold + 2 BM25 hard negatives per query + the other queries' documents; the query's other gold documents masked), learnable temperature,
40-minute budget, best dev-MAP checkpoint kept. Each run also scores the **untrained Phase-1 backbone** first (zero-shot transfer).

**3. Hybrid ranking.** Per query, the dense scores and BM25 scores are z-normalised and mixed: `w * dense + (1 - w) * BM25 - year penalty + popularity`,
weights tuned on dev; an *honest* estimate (weights fitted on half of the dev queries, scored on the other half) is printed for the hybrid, dense alone
and BM25 alone. Every run writes **two** submissions: `*_hybrid.zip` and `*_dense.zip` (the Phase-1 model alone).

| GPU 0 | GPU 1 |
|---|---|
| **LoRMA** (compulsory) - enc-dec encoder | QLoRA (NF4 base weights, from scratch) - RetNet |
| full fine-tuning - GQA | prompt tuning (20 soft tokens) - GQA |
| IA3 - MLA | partial fine-tuning (top 6 blocks) - MLA |
| **knowledge distillation**: 8-layer GQA student; teacher = mean of the 3 best runs' similarity scores on each training query's 60+ candidates (KL) + gold InfoNCE | ablation: LoRMA enc-dec with random instead of BM25 hard negatives |

then an **ensemble** (mean of the runs' z-scored similarities, members chosen greedily on the honest dev F1@7, fused with BM25), and the official
`evaluate_phase7.py` on every dev prediction. All zips are collected in `/kaggle/working/submissions/`. All adaptation methods are written from scratch
in `pcr_peft.py` (no `peft` / `bitsandbytes`).

In [1]:
import os
HF_TOKEN = None
for _name in ("HF", "HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(_name)
        if HF_TOKEN:
            break
    except Exception as e:
        print("secret", _name, "not available:", type(e).__name__)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("No HF token found")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
%%writefile pcr_metrics.py
"""Phase 7 (PCR) metrics, score fusion and output writing - numpy only.
Micro-F1@7 / P@7 / R@7 follow evaluate_phase7.py exactly: per query the top-7 ids are compared with the COMPLETE gold set
(gold ids missing from the pool still count as false negatives); tp, fp, fn are summed over queries.
Score matrices are (n_queries, n_pool) float arrays; -inf marks a candidate that must never be returned (the query itself)."""
import json, zipfile
import numpy as np

K = 7


def topk(S, k=K):
    """Indices of the k highest scores per row, best first (ties broken by column index for determinism)."""
    k = min(k, S.shape[1])
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    rows = np.arange(S.shape[0])[:, None]
    order = np.lexsort((part, -S[rows, part]), axis=1)
    return part[rows, order]


def gold_matrix(gold_lists, cids):
    """gold_lists: list (per query) of gold id lists. -> boolean (nq, nc) for golds inside the pool, and |gold| per query."""
    pos = {c: j for j, c in enumerate(cids)}
    G = np.zeros((len(gold_lists), len(cids)), dtype=bool)
    ng = np.zeros(len(gold_lists), dtype=np.int64)
    for i, g in enumerate(gold_lists):
        g = list(dict.fromkeys(x for x in g if isinstance(x, str) and x.strip()))
        ng[i] = len(g)
        for c in g:
            if c in pos:
                G[i, pos[c]] = True
    return G, ng


def prf_at_k(S, G, ng, k=K, rows=None):
    if rows is not None:
        S, G, ng = S[rows], G[rows], ng[rows]
    T = topk(S, k)
    tp = int(np.take_along_axis(G, T, 1).sum())
    fp = T.shape[0] * T.shape[1] - tp
    fn = int(ng.sum()) - tp
    p = tp / max(1, tp + fp)
    r = tp / max(1, tp + fn)
    f = 2 * tp / max(1, 2 * tp + fp + fn)
    return f, p, r


def f1_at_k(S, G, ng, k=K, rows=None):
    return prf_at_k(S, G, ng, k, rows)[0]


def ranking_stats(S, G, ng, ks=(7, 20, 50, 100)):
    """MAP (gold outside the pool counts as never retrieved), per-query (macro) recall R@k, MRR."""
    order = np.argsort(-S, axis=1, kind="stable")
    hit = np.take_along_axis(G, order, 1)
    ranks = np.arange(1, S.shape[1] + 1)
    ch = np.cumsum(hit, 1)
    ap = (hit * ch / ranks).sum(1) / np.maximum(ng, 1)
    first = np.where(hit.any(1), hit.argmax(1) + 1, np.inf)
    out = {"map": float(ap[ng > 0].mean()), "mrr": float((1.0 / first)[ng > 0].mean())}
    for k in ks:
        out[f"R@{k}"] = float((ch[:, min(k, S.shape[1]) - 1] / np.maximum(ng, 1))[ng > 0].mean())
    return out


def full_report(S, G, ng):
    f, p, r = prf_at_k(S, G, ng)
    rep = {"f1@7": f, "p@7": p, "r@7": r}
    rep.update(ranking_stats(S, G, ng))
    return rep


def fmt(rep):
    return " | ".join(f"{k} {v:.4f}" for k, v in rep.items() if isinstance(v, float))


def oracle_f1(ng, ng_in_pool, k=K):
    """Best possible micro-F1@k: every gold that is in the pool ranked first."""
    tp = np.minimum(ng_in_pool, k).sum()
    fp = k * len(ng) - tp
    fn = ng.sum() - tp
    return float(2 * tp / max(1, 2 * tp + fp + fn))


def zrow(S):
    """Per-query z-score over the finite entries; masked (-inf) entries stay -inf."""
    S = np.asarray(S, dtype=np.float32)
    fin = np.isfinite(S)
    X = np.where(fin, S, np.nan)
    mu = np.nanmean(X, 1, keepdims=True)
    sd = np.nanstd(X, 1, keepdims=True)
    Z = (X - mu) / np.maximum(sd, 1e-6)
    return np.where(fin, Z, -np.inf).astype(np.float32)


W_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 3)
PEN_GRID = (0.0, 0.5, 1.0, 2.0, 4.0)
POP_GRID = (0.0, 0.15, 0.3, 0.6)


def fuse(Zd, Zb, ybad, pop, w, pen, gp):
    """w * dense + (1 - w) * BM25 (both per-query z-scores) - pen * [candidate newer than query] + gp * log(1 + train citations).
    Zd may be None (lexical only)."""
    S = Zb * (1 - w) if Zd is None or w == 0 else (Zb * (1 - w) + Zd * w if w < 1 else Zd.copy())
    if pen:
        S = S - pen * ybad
    if gp:
        S = S + gp * pop[None, :]
    return S


def _grid(Zd):
    return [(float(w), p, g) for w in (W_GRID if Zd is not None else (0.0,)) for p in PEN_GRID for g in POP_GRID]


def tune(Zd, Zb, ybad, pop, G, ng, rows=None, grid=None):
    grid = grid or _grid(Zd)
    if rows is not None:
        Zd = None if Zd is None else Zd[rows]
        Zb, ybad, G, ng = Zb[rows], ybad[rows], G[rows], ng[rows]
    best = (-1.0, None)
    for w, p, g in grid:
        f = f1_at_k(fuse(Zd, Zb, ybad, pop, w, p, g), G, ng)
        if f > best[0] + 1e-12:
            best = (f, (w, p, g))
    return best[1], best[0]


def honest_tune(Zd, Zb, ybad, pop, G, ng, folds=2, reps=3, seed=0):
    """Fusion weights fitted on part of the dev queries and scored on the rest (repeated): an unbiased estimate of the
    hybrid's F1@7, plus the same estimate for the fixed choices 'dense only' and 'BM25 only'."""
    n = G.shape[0]
    res = {"tuned": [], "dense_only": [], "bm25_only": []}
    for rep in range(reps):
        idx = np.random.RandomState(seed + rep).permutation(n)
        parts = np.array_split(idx, folds)
        for k in range(folds):
            te = parts[k]
            tr = np.concatenate([parts[j] for j in range(folds) if j != k])
            prm, _ = tune(Zd, Zb, ybad, pop, G, ng, rows=tr)
            res["tuned"].append(f1_at_k(fuse(Zd, Zb, ybad, pop, *prm), G, ng, rows=te))
            if Zd is not None:
                res["dense_only"].append(f1_at_k(Zd, G, ng, rows=te))
            res["bm25_only"].append(f1_at_k(Zb, G, ng, rows=te))
    return {k: float(np.mean(v)) for k, v in res.items() if v}


def write_prediction(qids, cids, S, json_path, zip_path, k=K):
    """{query_id: [7 unique candidate ids, best first]}, prediction.json at the zip root."""
    T = topk(S, k)
    out = {}
    for i, q in enumerate(qids):
        lst = [cids[j] for j in T[i]]
        assert len(lst) == k and len(set(lst)) == k and q not in lst, f"bad ranking for {q}"
        out[q] = lst
    json.dump(out, open(json_path, "w"))
    if zip_path is not None:
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
            z.write(json_path, arcname="prediction.json")
    return out


Writing pcr_metrics.py


In [3]:
%%writefile pcr_prep.py
"""Phase 7 (PCR) data preparation - runs once on CPU before the GPU experiments. Usage: python pcr_prep.py [--smoke]
1. downloads and parses train/dev/test queries and their candidate pools (each split has its own pool)
2. data analysis: pool sizes, gold set sizes, gold coverage by the pool, the oracle F1@7, document lengths, overlaps
3. BM25 (written from scratch on sparse counts) with k1 / b / n-gram / query weighting tuned on dev
4. two non-neural signals checked on dev: (a) a candidate that mentions a year later than any year in the query cannot
   be a PRIOR case; (b) how often the candidate is cited by training queries (when pools overlap)
5. per training query: gold + BM25 top-60 (hard-negative list and the KD candidate list)
Writes pcr_cache/data.pkl, dev_truth.json, candidate_ids_{dev,test}.json and the BM25 baseline submission."""
import os, sys, re, json, time, pickle, hashlib
from pathlib import Path
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from pcr_metrics import gold_matrix, full_report, fmt, oracle_f1, zrow, honest_tune, tune, fuse, write_prediction, ranking_stats

SMOKE = "--smoke" in sys.argv
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
CACHE = ROOT / "pcr_cache"
SUB = ROOT / "submissions"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = os.environ.get("HF_TOKEN") or None
LIST_TOP = 60
YEAR_RE = re.compile(r"\b(1[89]\d\d|20[0-2]\d)\b")


# ------------------------------------------------------------------ parsing
def as_text(v):
    if isinstance(v, dict):
        for k in ("text", "document", "sentences", "content", "case_text"):
            if k in v:
                return as_text(v[k])
        return ""
    if isinstance(v, list):
        return " ".join(as_text(x) if not isinstance(x, str) else x for x in v)
    return "" if v is None else str(v)


def parse_queries(path):
    raw = json.load(open(path, encoding="utf-8"))
    recs = []
    if isinstance(raw, dict):
        items = list(raw.items())
        if len(items) == 1 and isinstance(items[0][1], list):
            raw = items[0][1]
        else:
            for k, v in items:
                gold = v.get("relevant_candidates") if isinstance(v, dict) else None
                recs.append((str(k), as_text(v), gold))
            return recs, raw
    for i, r in enumerate(raw):
        qid = str(r.get("id", r.get("query_id", r.get("qid", i))))
        recs.append((qid, as_text(r), r.get("relevant_candidates")))
    return recs, raw


def parse_pool(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        items = list(raw.items())
        if len(items) == 1 and isinstance(items[0][1], list) and items[0][1] and isinstance(items[0][1][0], dict):
            raw = items[0][1]
        else:
            return [(str(k), as_text(v)) for k, v in items], raw
    out = []
    for i, r in enumerate(raw):
        cid = str(r.get("id", r.get("candidate_id", r.get("cid", i)))) if isinstance(r, dict) else str(i)
        out.append((cid, as_text(r)))
    return out, raw


def clean_gold(g):
    if g is None:
        return None
    out = []
    for c in g:
        if isinstance(c, (int, np.integer)):
            c = str(c)
        if isinstance(c, str) and c.strip() and c not in out:
            out.append(c)
    return out


def norm(t):
    return " ".join(t.split())


def year_of(t):
    """Old estimate: the latest 4-digit year anywhere in the text (picks up amounts, section numbers and, in Supreme
    Court headers, the CITATOR INFO list of LATER judgments that cited this one)."""
    ys = [int(y) for y in YEAR_RE.findall(t)]
    ys = [y for y in ys if y <= 2026]
    return max(ys) if ys else -1


_MON = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*"
_Y = r"((?:18|19|20)\d\d)"
DATE_RES = [re.compile(p) for p in (
    r"\b\d{1,2}[./-]\d{1,2}[./-]" + _Y + r"\b",                                          # 07/05/1963
    r"\b" + _MON + r"\.?\s+\d{1,2}(?:st|nd|rd|th)?,?\s+" + _Y + r"\b",                       # November 25, 1924
    r"\b\d{1,2}(?:st|nd|rd|th)?\s+(?:day\s+of\s+)?" + _MON + r"\.?,?\s+" + _Y + r"\b",     # 25th day of November 1924
    r"\b(?:AIR|ILR|ITR|SCR|SCC|SCJ|Cri\.?\s?L\.?\s?J\.?|Mad|Bom|Cal|All|Pat|Lah)\s*\(?" + _Y + r"\b",  # AIR 1954 SC 300
    r"\(" + _Y + r"\)\s*\d*\s*(?:SCR|SCC|ITR|SCJ|MLJ)\b",                                   # (1955) 1 SCR 1
    r"\b" + _Y + r"\s+(?:AIR|SCR|SCC)\b")]                                                     # 1964 AIR 1545
JUDG_RE = re.compile(r"DATE OF JUDGMENT\W*\d{1,2}[./-]\d{1,2}[./-]((?:18|19|20)\d\d)")
CITATOR_RE = re.compile(r"CITATOR INFO.*?(?=ACT:|HEADNOTE:|JUDGMENT:|$)", re.S)


def dated_year(t, query=False):
    """New estimate: the judgment date in Supreme Court headers ("DATE OF JUDGMENT: 07/05/1963"); otherwise the latest
    year that appears inside a date or a law-report citation (amounts and section numbers ignored), with the
    CITATOR INFO block removed. -1 when nothing date-like is found (then no year penalty is applied to that pair)."""
    if query:
        m = JUDG_RE.search(t[:3000])
        if m:
            return int(m.group(1))
    t = CITATOR_RE.sub(" ", t[:200000])
    ys = [int(y) for r in DATE_RES for y in r.findall(t)]
    ys = [y for y in ys if y <= 2026]
    return max(ys) if ys else -1


# ------------------------------------------------------------------ BM25
def bm25_weights(C, k1, b):
    """C: (nc, V) csr term counts of the pool. Returns csr of tf-saturated weights and the idf vector."""
    C = C.tocsr().astype(np.float32)
    N = C.shape[0]
    df = np.bincount(C.indices, minlength=C.shape[1]).astype(np.float32)
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(C.sum(1)).ravel()
    avg = max(1.0, dl.mean())
    rows = np.repeat(np.arange(N), np.diff(C.indptr))
    tf = C.data
    W = C.copy()
    W.data = tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avg))
    return W, idf


def bm25_scores(Q, C, k1, b, qmode):
    W, idf = bm25_weights(C, k1, b)
    Q = Q.tocsr().astype(np.float32)
    Qw = Q.copy()
    Qw.data = (np.log1p(Q.data) if qmode == "log" else np.ones_like(Q.data))
    Qw = Qw.multiply(idf[None, :]).tocsr()
    return np.asarray((Qw @ W.T).todense(), dtype=np.float32)


def vectorize(ngram, pool_text, query_text):
    vec = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_]+\b|\b\d{1,4}\b", ngram_range=ngram,
                          min_df=2, max_df=0.6, stop_words="english", dtype=np.int32)
    C = vec.fit_transform(pool_text)
    Q = vec.transform(query_text)
    return C, Q, len(vec.vocabulary_)


def main():
    t00 = time.time()
    CACHE.mkdir(parents=True, exist_ok=True)
    SUB.mkdir(parents=True, exist_ok=True)
    from huggingface_hub import hf_hub_download, HfApi
    files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    p7 = [f for f in files if "Phase_7" in f or "PCR" in f]
    print("Phase-7 files:", p7, flush=True)

    def fetch(name):
        m = [f for f in p7 if f.endswith("/" + name) or f == name]
        assert m, f"{name} not found in {p7}"
        return hf_hub_download(HF_DATASET_REPO, m[0], repo_type="dataset", token=HF_TOKEN)

    data = {}
    for split in ("train", "dev", "test"):
        q, rawq = parse_queries(fetch(f"{split}.json"))
        c, rawc = parse_pool(fetch(f"{split}_candidates.json"))
        ex = rawq[next(iter(rawq))] if isinstance(rawq, dict) else rawq[0]
        exc = rawc[next(iter(rawc))] if isinstance(rawc, dict) else rawc[0]
        print(f"\n{split}: {len(q)} queries | raw {type(rawq).__name__}, example keys "
              f"{list(ex.keys()) if isinstance(ex, dict) else type(ex).__name__} | pool {len(c)} docs, raw {type(rawc).__name__}, "
              f"example {list(exc.keys()) if isinstance(exc, dict) else type(exc).__name__}")
        print(f"  query {q[0][0]!r}: {norm(q[0][1])[:300]!r}")
        print(f"  candidate {c[0][0]!r}: {norm(c[0][1])[:300]!r}")
        # unique ids (keep the first occurrence)
        seen, qq = set(), []
        for r in q:
            if r[0] not in seen:
                seen.add(r[0]); qq.append(r)
        seen, cc = set(), []
        for r in c:
            if r[0] not in seen:
                seen.add(r[0]); cc.append(r)
        if len(qq) != len(q) or len(cc) != len(c):
            print(f"  duplicate ids removed: queries {len(q) - len(qq)}, candidates {len(c) - len(cc)}")
        qids = [r[0] for r in qq]
        qtext = [norm(r[1]) for r in qq]
        gold = [clean_gold(r[2]) for r in qq]
        cids = [r[0] for r in cc]
        ctext = [norm(r[1]) for r in cc]
        d = dict(qids=qids, qtext=qtext, cids=cids, ctext=ctext, has_gold=split != "test")
        if split != "test":
            assert all(g is not None for g in gold), f"{split}: queries without relevant_candidates"
            d["gold"] = gold
        data[split] = d

    # ---------------------------------------------------------- analysis
    print("\n=== data analysis ===")
    for s, d in data.items():
        cpos = {c: j for j, c in enumerate(d["cids"])}
        qh = [hashlib.md5(t[:5000].encode("utf-8", "ignore")).hexdigest() for t in d["qtext"]]
        ch = {}
        for j, t in enumerate(d["ctext"]):
            ch.setdefault(hashlib.md5(t[:5000].encode("utf-8", "ignore")).hexdigest(), []).append(j)
        mask = np.zeros((len(d["qids"]), len(d["cids"])), dtype=bool)
        n_self = n_dup = 0
        for i, q in enumerate(d["qids"]):
            if q in cpos:
                mask[i, cpos[q]] = True; n_self += 1
            for j in ch.get(qh[i], []):
                if not mask[i, j]:
                    mask[i, j] = True; n_dup += 1
        d["mask"] = mask
        qw = np.array([len(t.split()) for t in d["qtext"]]); cw = np.array([len(t.split()) for t in d["ctext"]])
        msg = (f"{s}: {len(d['qids'])} queries, pool {len(d['cids'])} | query words median {np.median(qw):.0f} p90 {np.percentile(qw, 90):.0f} "
               f"| candidate words median {np.median(cw):.0f} p90 {np.percentile(cw, 90):.0f} max {cw.max()} | empty texts q {int((qw == 0).sum())} "
               f"c {int((cw == 0).sum())} | query id inside its own pool {n_self} | candidate text identical to the query {n_dup}")
        if d["has_gold"]:
            G, ng = gold_matrix(d["gold"], d["cids"])
            d["G"], d["ng"] = G, ng
            inpool = G.sum(1)
            gl = ng[ng > 0]
            msg += (f"\n   gold per query: mean {ng.mean():.2f} median {np.median(ng):.0f} max {ng.max()} | queries with no gold {int((ng == 0).sum())} "
                    f"| with > 7 gold {int((ng > 7).sum())} ({100 * np.mean(ng > 7):.0f}%) | gold ids missing from the pool "
                    f"{int(ng.sum() - inpool.sum())} of {int(ng.sum())} | oracle F1@7 {oracle_f1(ng, inpool):.4f}"
                    f"\n   gold size histogram {np.bincount(np.minimum(gl, 15)).tolist()} (last bin = 15+)"
                    f" | gold pairs pointing at the query itself {int((G & mask).sum())}")
        print(msg)
    print("pool overlaps (shared candidate ids): train-dev", len(set(data["train"]["cids"]) & set(data["dev"]["cids"])),
          "| train-test", len(set(data["train"]["cids"]) & set(data["test"]["cids"])),
          "| dev-test", len(set(data["dev"]["cids"]) & set(data["test"]["cids"])),
          "| test queries also train/dev queries", len(set(data["test"]["qids"]) & (set(data["train"]["qids"]) | set(data["dev"]["qids"]))))
    # popularity: how many training queries cite the candidate
    cnt = {}
    for g in data["train"]["gold"]:
        for c in g:
            cnt[c] = cnt.get(c, 0) + 1
    for s in ("dev", "test"):
        d = data[s]
        d["pop"] = np.log1p(np.array([cnt.get(c, 0) for c in d["cids"]], dtype=np.float32))
        msg = f"{s}: pool candidates cited by >= 1 training query: {int((d['pop'] > 0).sum())} of {len(d['cids'])}"
        if d["has_gold"]:
            gp = np.concatenate([d["pop"][d["G"][i]] for i in range(len(d["qids"]))]) if d["G"].any() else np.zeros(1)
            msg += f" | share of dev gold pairs whose candidate is cited in train: {np.mean(gp > 0):.2f} (pool share {np.mean(d['pop'] > 0):.2f})"
        print(msg)
    data["train"]["pop"] = np.zeros(len(data["train"]["cids"]), np.float32)
    # years: a prior case cannot be newer than the query. Two estimates, the one that separates gold from non-gold
    # pairs better on dev (lower share of gold pairs flagged relative to all pairs flagged) is used.
    def flag(qy, cy):
        return (cy[None, :] > qy[:, None]) & (qy[:, None] > 0) & (cy[None, :] > 0)

    YM = {"max_year": (lambda t: year_of(t), lambda t: year_of(t)),
          "dated": (lambda t: dated_year(t, query=True), lambda t: dated_year(t))}
    years, sep = {}, {}
    for name, (fq, fc) in YM.items():
        for s, d in data.items():
            qy, cy = np.array([fq(t) for t in d["qtext"]]), np.array([fc(t) for t in d["ctext"]])
            years[name, s] = (qy, cy)
            yb = flag(qy, cy)
            msg = (f"{s} [{name}]: query year median {np.median(qy[qy > 0]) if (qy > 0).any() else -1:.0f} (unknown {int((qy < 0).sum())}) | "
                   f"candidate year median {np.median(cy[cy > 0]) if (cy > 0).any() else -1:.0f} (unknown {int((cy < 0).sum())}) | "
                   f"pairs flagged 'candidate newer than query': {yb.mean():.3f}")
            if d["has_gold"]:
                g = yb[d["G"]].mean() if d["G"].any() else 0.0
                msg += f" | among GOLD pairs: {g:.3f}"
                if s == "dev":
                    sep[name] = g / yb.mean() if yb.mean() >= 0.05 else float("inf")    # must flag a useful share of pairs
            print(msg)
    YEAR_METHOD = min(sep, key=lambda k: (sep[k], k != "max_year"))
    print(f"year estimate used: {YEAR_METHOD} (gold-flag rate / all-flag rate on dev: "
          + ", ".join(f"{k} {v:.3f}" for k, v in sep.items()) + ")")
    for s, d in data.items():
        d["qyear"], d["cyear"] = years[YEAR_METHOD, s]

    # ---------------------------------------------------------- BM25 on dev
    print("\n=== BM25 tuning on dev ===", flush=True)
    dv = data["dev"]
    ybad_dev = ((dv["cyear"][None, :] > dv["qyear"][:, None]) & (dv["qyear"][:, None] > 0) & (dv["cyear"][None, :] > 0)).astype(np.float32)
    grid_ng = [(1, 1)] if SMOKE else [(1, 1), (1, 2)]
    grid_kb = [(1.2, 0.75)] if SMOKE else [(0.9, 0.4), (1.2, 0.75), (1.5, 0.9), (2.0, 0.75)]
    best, cfgs = None, []
    for ngram in grid_ng:
        t0 = time.time()
        C, Q, V = vectorize(ngram, dv["ctext"], dv["qtext"])
        for k1, b in grid_kb:
            for qmode in ("log", "bin"):
                S = bm25_scores(Q, C, k1, b, qmode)
                S[dv["mask"]] = -np.inf
                rep = full_report(S, dv["G"], dv["ng"])
                cfgs.append(dict(ngram=list(ngram), k1=k1, b=b, qmode=qmode, **rep))
                print(f"  ngram {ngram} k1 {k1} b {b} q {qmode:>3} (vocab {V}): {fmt(rep)}", flush=True)
                if best is None or rep["map"] > best[0]["map"]:
                    best = (cfgs[-1], S)
        print(f"  ({time.time() - t0:.0f}s)")
    BM = {k: best[0][k] for k in ("ngram", "k1", "b", "qmode")}
    print("chosen BM25 config (best dev MAP):", BM, "|", fmt({k: v for k, v in best[0].items() if k not in ("k1", "b")}))

    # ---------------------------------------------------------- BM25 everywhere with the chosen config
    for s, d in data.items():
        t0 = time.time()
        C, Q, V = vectorize(tuple(BM["ngram"]), d["ctext"], d["qtext"])
        S = bm25_scores(Q, C, BM["k1"], BM["b"], BM["qmode"])
        S[d["mask"]] = -np.inf
        d["bm25"] = S
        print(f"{s}: BM25 {S.shape} in {time.time() - t0:.0f}s" + (f" | {fmt(full_report(S, d['G'], d['ng']))}" if d["has_gold"] else ""), flush=True)
        d["ybad"] = ((d["cyear"][None, :] > d["qyear"][:, None]) & (d["qyear"][:, None] > 0) & (d["cyear"][None, :] > 0)).astype(np.float32)
    # training lists: gold (in pool) + BM25 top-LIST_TOP
    tr = data["train"]
    order = np.argsort(-tr["bm25"], axis=1)[:, :LIST_TOP]
    lists, n_pos = [], []
    for i in range(len(tr["qids"])):
        g = np.where(tr["G"][i])[0]
        lst = list(dict.fromkeys(list(g) + [int(j) for j in order[i] if not tr["mask"][i, j]]))
        lists.append(np.asarray(lst, dtype=np.int32)); n_pos.append(len(g))
    tr["lists"] = lists
    rs = ranking_stats(tr["bm25"], tr["G"], tr["ng"])
    print(f"train lists: {LIST_TOP} BM25 candidates + gold | train queries with >= 1 gold in pool {int(np.sum(np.array(n_pos) > 0))} of {len(lists)} "
          f"| BM25 recall@50 on train {rs['R@50']:.3f}, @100 {rs['R@100']:.3f}")

    # ---------------------------------------------------------- lexical baseline + fusion signals
    Zb = zrow(dv["bm25"])
    h = honest_tune(None, Zb, dv["ybad"], dv["pop"], dv["G"], dv["ng"])
    prm, f = tune(None, Zb, dv["ybad"], dv["pop"], dv["G"], dv["ng"])
    print(f"\nlexical baseline: BM25 alone dev F1@7 {h['bm25_only']:.4f} | BM25 + year penalty + popularity: tuned {f:.4f}, "
          f"honest (fitted on half of dev, scored on the other half) {h['tuned']:.4f} | weights (w_dense, year_pen, pop) {prm}")
    ds = data["test"]
    write_prediction(dv["qids"], dv["cids"], fuse(None, Zb, dv["ybad"], dv["pop"], *prm), ROOT / "dev_prediction_bm25.json", None)
    write_prediction(ds["qids"], ds["cids"], fuse(None, zrow(ds["bm25"]), ds["ybad"], ds["pop"], *prm),
                     ROOT / "prediction_bm25.json", SUB / f"{'smoke_' if SMOKE else ''}baseline_bm25_lexical.zip")
    json.dump({q: g for q, g in zip(dv["qids"], dv["gold"])}, open(ROOT / "dev_truth.json", "w"))
    json.dump(dv["cids"], open(ROOT / "candidate_ids_dev.json", "w"))
    json.dump(ds["cids"], open(ROOT / "candidate_ids_test.json", "w"))
    meta = dict(bm25=BM, bm25_grid=cfgs, year_method=YEAR_METHOD, year_separation=sep, lexical_fusion=dict(params=prm, dev_f1=f, honest=h), list_top=LIST_TOP, smoke=SMOKE)
    with open(CACHE / "data.tmp", "wb") as fh:
        pickle.dump(dict(data=data, meta=meta), fh, protocol=4)
    os.replace(CACHE / "data.tmp", CACHE / "data.pkl")
    json.dump(meta, open(CACHE / "meta.json", "w"), indent=1, default=float)
    print(f"\nPREP_DONE in {(time.time() - t00) / 60:.1f} min")


if __name__ == "__main__":
    main()


Writing pcr_prep.py


In [4]:
%%writefile pcr_lib.py
"""CS-781 Phase 7 library: prior-case retrieval with a Phase-1 backbone as a bi-encoder over long judgments.
document -> BPE tokens -> 512-token chunks; up to STORE chunks evenly spaced over the whole judgment are kept
-> every chunk goes through the backbone (decoders: causal states; enc-dec: bidirectional encoder with a padding mask)
-> masked mean pooling -> LayerNorm + linear projection to 256 dims -> L2 normalisation  = one vector per chunk.
Relevance(query, candidate) = late interaction over chunks ("chunk max-sim"): for every query chunk take its best-matching
candidate chunk, average over the query chunks. Single-vector cosine of the mean chunk vectors is reported alongside.
Training: InfoNCE over the batch (sampled gold + BM25 hard negatives of every query + the other queries' documents,
other gold documents of the query masked), learnable temperature. KD: + KL to the teacher's score distribution."""
import os, sys, json, math, time, random, hashlib, types, pickle, importlib.util, contextlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PAD_ID = 1


# ====================================================================== Phase-1 models
def load_module_from_file(path, name):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()[:10]


def find_p1_export(arch, roots=("/kaggle/input", "/kaggle/working")):
    for r in roots:
        for root, dirs, files in os.walk(r, followlinks=True):
            if "config.json" in files and "model.pt" in files:
                try:
                    if json.load(open(os.path.join(root, "config.json"))).get("arch") == arch:
                        return Path(root)
                except Exception:
                    pass
    return None


def load_p1(export_dir):
    export_dir = Path(export_dir)
    cj = json.load(open(export_dir / "config.json"))
    mod = load_module_from_file(export_dir / "legal_arch.py", f"legal_arch_{cj['arch']}_{file_hash(export_dir / 'legal_arch.py')}")
    bb = mod.build_model(mod.ArchConfig(**cj["cfg"]))
    bb.load_state_dict(torch.load(export_dir / "model.pt", map_location="cpu", weights_only=False))
    return mod, cj["cfg"], bb


def blocks_of(bb):
    return bb.blocks if bb.kind == "decoder" else bb.enc_blocks


def final_norm_of(bb):
    return bb.norm_f if bb.kind == "decoder" else bb.enc_norm


def strip_backbone(bb):
    """Retrieval needs only the encoder side: the enc-dec decoder and every LM head are dropped."""
    if bb.kind == "encdec":
        del bb.dec_blocks
        del bb.dec_norm
    bb.lm_head = nn.Identity()
    return bb


def replace_modules(bb, match, factory):
    n = 0
    for blk in blocks_of(bb):
        for name, mod in list(blk.named_modules()):
            if name and match(name, mod):
                parent_name, _, leaf = name.rpartition(".")
                parent = blk.get_submodule(parent_name) if parent_name else blk
                setattr(parent, leaf, factory(mod))
                n += 1
    return n


def replace_linears(bb, targets, factory):
    targets = set(targets)
    return replace_modules(bb, lambda n, m: isinstance(m, nn.Linear) and n.split(".")[-1] in targets, factory)


def make_student_bb(mod, cfg, full_bb, keep):
    cfg = dict(cfg)
    if full_bb.kind == "decoder":
        cfg["n_layers"] = len(keep)
    else:
        cfg["n_enc_layers"] = len(keep)
        cfg["n_dec_layers"] = 1
    st = mod.build_model(mod.ArchConfig(**cfg))
    st.tok_emb.load_state_dict(full_bb.tok_emb.state_dict())
    final_norm_of(st).load_state_dict(final_norm_of(full_bb).state_dict())
    for i, j in enumerate(keep):
        blocks_of(st)[i].load_state_dict(blocks_of(full_bb)[j].state_dict())
    return st, cfg


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


def max_positions(bb):
    rc = getattr(bb, "rope_cos", None)
    return int(rc.shape[0]) if rc is not None else int(getattr(bb.cfg, "max_seq_len", 1024))


# ---- key-padding mask for the enc-dec encoder (the Phase-1 encoder never saw padding)
_ENC_MASK = [None]


def _masked_enc_attn_forward(self, x, cos, sin, cache=None):
    arch = sys.modules[type(self).__module__]
    B, T, _ = x.shape
    q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    k = self.wk(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    v = self.wv(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    q = arch.apply_rope(q, cos[:T], sin[:T])
    k = arch.apply_rope(k, cos[:T], sin[:T])
    m = _ENC_MASK[0]
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if m is None else m[:, None, None, :T])
    return self.wo(o.transpose(1, 2).reshape(B, T, -1))


def patch_encoder_padding_mask(bb):
    if bb.kind != "encdec":
        return 0
    for blk in bb.enc_blocks:
        blk.attn.forward = types.MethodType(_masked_enc_attn_forward, blk.attn)
    return len(bb.enc_blocks)


def _call_block(blk, h, cos, sin, mask):
    if mask is not None:
        _ENC_MASK[0] = mask                 # set again when checkpointing recomputes the block in backward
    return blk(h, cos, sin)


def run_backbone(bb, x, valid, train_from=0, ckpt=True):
    """x, valid: (N, T). Prompt tuning: bb.prompt (P, d) soft tokens are prepended and removed again at the output."""
    enc = bb.kind == "encdec"
    blocks, norm = blocks_of(bb), final_norm_of(bb)
    cos, sin = bb.rope_cos, bb.rope_sin
    grad = torch.is_grad_enabled()
    prompt = getattr(bb, "prompt", None)
    try:
        with torch.set_grad_enabled(grad and bb.tok_emb.weight.requires_grad and train_from == 0):
            h = bb.tok_emb(x)
        P = 0
        if prompt is not None:
            P = prompt.shape[0]
            h = torch.cat([prompt.to(h.dtype).unsqueeze(0).expand(h.shape[0], -1, -1), h], 1)
            valid = torch.cat([valid.new_ones(valid.shape[0], P), valid], 1)
        if enc:
            _ENC_MASK[0] = valid
        for i, blk in enumerate(blocks):
            if grad and i >= train_from:
                h = checkpoint(_call_block, blk, h, cos, sin, valid if enc else None, use_reentrant=False) if ckpt \
                    else _call_block(blk, h, cos, sin, valid if enc else None)
            else:
                with torch.no_grad():
                    h = _call_block(blk, h, cos, sin, valid if enc else None)
        h = norm(h)
        return h[:, P:] if P else h
    finally:
        if enc:
            _ENC_MASK[0] = None


# ====================================================================== tokens and chunks
def tokenize_docs(texts, tokenizer, chunk, store, cache_path=None):
    """Every document -> up to `store` chunks of `chunk` tokens, evenly spaced over the WHOLE document
    (always the first and the last chunk). Returns a list of lists of uint16/int32 arrays + the full token counts."""
    if cache_path is not None and Path(cache_path).exists():
        return pickle.load(open(cache_path, "rb"))
    dt = np.uint16 if tokenizer.get_vocab_size() < 65535 else np.int32
    docs, ntok = [], []
    for i in range(0, len(texts), 500):
        enc = tokenizer.encode_batch([t if t else "." for t in texts[i:i + 500]])
        for e in enc:
            ids = np.asarray(e.ids or [PAD_ID], dtype=dt)
            n = max(1, math.ceil(len(ids) / chunk))
            sel = range(n) if n <= store else sorted(set(np.round(np.linspace(0, n - 1, store)).astype(int).tolist()))
            # the final chunk is the document's last `chunk` tokens (not a short remainder)
            docs.append([ids[j * chunk:(j + 1) * chunk] if j < n - 1 else ids[max(0, len(ids) - chunk):] for j in sel])
            ntok.append(len(ids))
    out = (docs, np.asarray(ntok))
    if cache_path is not None:
        tmp = f"{cache_path}.{os.getpid()}.tmp"
        pickle.dump(out, open(tmp, "wb"), protocol=4)
        os.replace(tmp, cache_path)
    return out


def pick_chunks(chunks, k, rng=None):
    """Inference: k evenly spaced stored chunks. Training (rng given): k random stored chunks, kept in document order."""
    n = len(chunks)
    if n <= k:
        return chunks
    if rng is None:
        idx = sorted(set(np.round(np.linspace(0, n - 1, k)).astype(int).tolist()))
    else:
        idx = sorted(rng.sample(range(n), k))
    return [chunks[j] for j in idx]


def chunk_batch(doc_chunk_lists):
    """[[chunk arrays] per doc] -> x (Nchunks, T), valid (Nchunks, T), owner doc index per chunk, chunks per doc."""
    rows, owner = [], []
    for d, cl in enumerate(doc_chunk_lists):
        for c in cl:
            rows.append(c if len(c) else np.asarray([PAD_ID]))
            owner.append(d)
    T = max(len(r) for r in rows)
    x = np.full((len(rows), T), PAD_ID, dtype=np.int64)
    v = np.zeros((len(rows), T), dtype=bool)
    for i, r in enumerate(rows):
        x[i, :len(r)] = r
        v[i, :len(r)] = True
    return torch.from_numpy(x), torch.from_numpy(v), np.asarray(owner), [len(cl) for cl in doc_chunk_lists]


def group(vecs, counts):
    """(Nchunks, D) in doc order -> (ndocs, Kmax, D) + mask (ndocs, Kmax)."""
    K = max(counts)
    D = vecs.shape[-1]
    out = vecs.new_zeros(len(counts), K, D)
    m = torch.zeros(len(counts), K, dtype=torch.bool, device=vecs.device)
    s = 0
    for i, c in enumerate(counts):
        out[i, :c] = vecs[s:s + c]
        m[i, :c] = True
        s += c
    return out, m


# ====================================================================== model
class PCRModel(nn.Module):
    def __init__(self, bb, dim=256, train_from=0, ckpt=True, tau=0.05):
        super().__init__()
        d = bb.cfg.d_model
        self.bb, self.train_from, self.ckpt = bb, train_from, ckpt
        self.proj = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, dim))
        self.log_tau = nn.Parameter(torch.tensor(math.log(tau)))

    def chunk_vectors(self, x, valid, raw=False):
        h = run_backbone(self.bb, x, valid, self.train_from, self.ckpt)
        m = valid.unsqueeze(-1).to(h.dtype)
        pooled = ((h * m).sum(1) / m.sum(1).clamp(min=1)).float()
        if raw:                                                     # zero-shot Phase-1 representation, no projection
            return F.normalize(pooled, dim=-1)
        return F.normalize(self.proj(pooled), dim=-1)

    def tau(self):
        return self.log_tau.exp().clamp(0.01, 0.5)


def maxsim(qv, qm, dv, dm):
    """qv (B, Kq, D), dv (M, Kc, D) -> (B, M): mean over valid query chunks of the max over valid candidate chunks."""
    s = torch.einsum("bid,mjd->bmij", qv, dv)
    s = s.masked_fill(~dm[None, :, None, :], -4.0).amax(-1)
    qmf = qm[:, None, :].float()
    return (s * qmf).sum(-1) / qmf.sum(-1).clamp(min=1)


def meansim(qv, qm, dv, dm):
    qa = F.normalize((qv * qm.unsqueeze(-1)).sum(1), dim=-1)
    da = F.normalize((dv * dm.unsqueeze(-1)).sum(1), dim=-1)
    return qa @ da.t()


# ====================================================================== AMP
def get_amp(device):
    if device == "cuda":
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        return (lambda: torch.autocast(device_type="cuda", dtype=dt)), dt
    return (lambda: contextlib.nullcontext()), None


def make_scaler(device, amp_dtype):
    en = device == "cuda" and amp_dtype == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=en)


# ====================================================================== inference
@torch.no_grad()
def encode_docs(model, docs, k, device, amp, bs=48, raw=False):
    """docs: list of chunk lists. Returns padded (n, Kmax, D) float16 chunk vectors on the CPU and the mask."""
    model.eval()
    flat = [(d, c) for d, cl in enumerate(docs) for c in pick_chunks(cl, k)]
    order = sorted(range(len(flat)), key=lambda i: -len(flat[i][1]))
    D = None
    vecs = None
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        x, v, _, _ = chunk_batch([[flat[i][1]] for i in idx])
        with amp():
            e = model.chunk_vectors(x.to(device), v.to(device), raw=raw).float()
        if vecs is None:
            D = e.shape[-1]
            vecs = torch.zeros(len(flat), D)
        vecs[idx] = e.cpu()
    counts = np.bincount([d for d, _ in flat], minlength=len(docs)).tolist()
    V, M = group(vecs, counts)
    model.train()
    return V.half(), M


@torch.no_grad()
def score_matrix(QV, QM, CV, CM, device, kind="maxsim", qbs=16):
    CVd, CMd = CV.to(device).float(), CM.to(device)
    out = []
    for s in range(0, QV.shape[0], qbs):
        qv, qm = QV[s:s + qbs].to(device).float(), QM[s:s + qbs].to(device)
        out.append((maxsim(qv, qm, CVd, CMd) if kind == "maxsim" else meansim(qv, qm, CVd, CMd)).float().cpu())
    return torch.cat(out).numpy()


# ====================================================================== training
def _log(s):
    print(s, flush=True)


def train(model, S, cfg, device, amp, amp_dtype, eval_fn, teacher=None, log=_log):
    """S: prepared training data (qdocs, cdocs, gold sets, hard-negative lists, pool size).
    Each step: BS queries; per query 1 sampled gold + N_NEG negatives (BM25 hard negatives, or random pool documents for
    the ablation); InfoNCE over all documents of the step with the query's other gold documents masked."""
    rng = random.Random(cfg["seed"])
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    bbp = [p for n, p in named if n.startswith("bb.")]
    hp = [p for n, p in named if not n.startswith("bb.")]
    groups = ([dict(params=bbp, lr=cfg["lr"], base_lr=cfg["lr"], weight_decay=cfg["wd"])] if bbp else []) + \
             [dict(params=hp, lr=cfg["head_lr"], base_lr=cfg["head_lr"], weight_decay=0.0)]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)
    scaler = make_scaler(device, amp_dtype)
    qidx = [i for i in range(len(S["qdocs"])) if S["gold"][i]]
    spe = math.ceil(len(qidx) / cfg["bs"])
    total = spe * cfg["epochs"]
    warm = max(1, int(cfg["warmup"] * min(total, 600)))
    keep = {n for n, _ in named}
    tot, trn = count_params(model)
    log(f"{len(qidx)} training queries with gold in the pool | {spe} steps/epoch x {cfg['epochs']} = {total} steps (budget {cfg['max_min']} min) | "
        f"params {tot / 1e6:.1f}M trainable {trn / 1e6:.3f}M ({100 * trn / tot:.2f}%)")
    hist, best, best_state, step, t0 = [], -1.0, None, 0, time.time()
    model.train()
    stop = False
    run = dict(loss=0.0, kd=0.0, n=0)
    for ep in range(cfg["epochs"]):
        order = qidx[:]
        rng.shuffle(order)
        for b0 in range(0, len(order), cfg["bs"]):
            qs = order[b0:b0 + cfg["bs"]]
            docs, own = [], []                                   # candidate indices in this step, first = sampled gold
            for q in qs:
                pos = rng.choice(S["gold"][q])
                if cfg["neg_mode"] == "random":
                    negs = []
                    while len(negs) < cfg["n_neg"]:
                        j = rng.randrange(S["n_pool"])
                        if j not in S["gold_set"][q] and j != pos and j not in negs and j not in S["excl"][q]:
                            negs.append(j)
                else:
                    cand = [j for j in S["lists"][q] if j not in S["gold_set"][q]]
                    negs = rng.sample(cand, min(cfg["n_neg"], len(cand)))
                mine = [pos] + negs
                own.append(mine)
                docs += mine
            uniq = list(dict.fromkeys(docs))
            col = {d: i for i, d in enumerate(uniq)}
            qx, qv_, _, qc = chunk_batch([pick_chunks(S["qdocs"][q], cfg["k_train"], rng) for q in qs])
            cx, cv_, _, cc = chunk_batch([pick_chunks(S["cdocs"][d], cfg["k_train"], rng) for d in uniq])
            with amp():
                qe = model.chunk_vectors(qx.to(device), qv_.to(device))
                ce = model.chunk_vectors(cx.to(device), cv_.to(device))
            QV, QM = group(qe, qc)
            CV, CM = group(ce, cc)
            sim = maxsim(QV.float(), QM, CV.float(), CM)         # (B, M) in [-1, 1]
            logits = sim / model.tau()
            mask = torch.zeros_like(logits, dtype=torch.bool)
            tgt = []
            for b, q in enumerate(qs):
                for d in uniq:
                    if d in S["gold_set"][q] and d != own[b][0]:
                        mask[b, col[d]] = True                   # another gold document of this query: not a negative
                tgt.append(col[own[b][0]])
            logits = logits.masked_fill(mask, -1e4)
            loss = F.cross_entropy(logits, torch.tensor(tgt, device=device))
            kdl = torch.zeros((), device=device)
            if teacher is not None:
                parts = []
                for b, q in enumerate(qs):
                    cols = [col[d] for d in own[b]]
                    t = torch.tensor([teacher[q].get(d, np.nan) for d in own[b]], dtype=torch.float32, device=device)
                    ok = torch.isfinite(t)
                    if ok.sum() < 2:
                        continue
                    s_log = F.log_softmax(logits[b, cols][ok], -1)
                    p_t = F.softmax(t[ok] / cfg["kd_T"], -1)
                    parts.append(F.kl_div(s_log, p_t, reduction="sum"))
                if parts:
                    kdl = torch.stack(parts).mean()
                loss = cfg["kd_alpha"] * loss + (1 - cfg["kd_alpha"]) * kdl
            if not torch.isfinite(loss):
                log("  non-finite loss - batch skipped")
                opt.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            run["loss"] += float(loss); run["kd"] += float(kdl); run["n"] += 1
            el = (time.time() - t0) / 60
            prog = max((step - warm) / max(1, total - warm), el / cfg["max_min"])
            mult = (step + 1) / warm if step < warm else max(0.03, 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * mult
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _, p in named], cfg["grad_clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % 20 == 0:
                log(f"  ep {ep + 1} step {step}/{total} loss {run['loss'] / max(1, run['n']):.4f}"
                    + (f" (kd {run['kd'] / max(1, run['n']):.4f})" if teacher is not None else "")
                    + f" | tau {float(model.tau()):.4f} | lr x{mult:.2f} | {el:.1f} min")
                run = dict(loss=0.0, kd=0.0, n=0)
            stop = el > cfg["max_min"]
            if step % cfg["eval_every"] == 0 or stop or step == total:
                rep = eval_fn(model)
                rec = dict(epoch=ep + 1, step=step, minutes=el, **rep)
                if rep["map"] > best:
                    best = rep["map"]
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep}
                    rec["best"] = True
                hist.append(rec)
                log("== " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
                model.train()
            if stop:
                log(f"time budget ({cfg['max_min']} min) reached - stopping")
                break
        if stop:
            break
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        log(f"restored best checkpoint (dev MAP {best:.4f})")
    del opt
    return hist, best


Writing pcr_lib.py


In [5]:
%%writefile pcr_peft.py
"""Adaptation methods for Phase 7, written from scratch (no `peft` / `bitsandbytes`). apply(method, bb, cfg) -> info dict.
All of them leave the retrieval head (LayerNorm + projection + temperature) trainable.
  full    : every backbone weight trainable
  partial : only the top `n_top` blocks + final norm trainable (the blocks below run without autograd)
  qlora   : frozen base weights stored in 4-bit NormalFloat (NF4, block 64, absmax per block, two codes per byte),
            de-quantised on the fly, LoRA adapters on top                                  (Dettmers et al., 2023)
  lorma   : (I + (alpha/r) B A) W0 x - multiplicative low-rank update                         (Bihany, Patel, Modi, 2025) [compulsory]
  ia3     : learned rescaling vectors on keys, values (or the MLA latent up-projection) and the FFN hidden
            activations, initialised to 1                                                         (Liu et al., 2022)
  prompt  : n_prompt soft tokens prepended to every chunk, initialised from random vocabulary embeddings (Lester et al., 2021)
  kd      : student backbone (every other layer of the GQA model), fully fine-tuned on gold + teacher score distributions"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pcr_lib import replace_linears, replace_modules, blocks_of, final_norm_of

DEFAULT_TARGETS = ("wq", "wk", "wv", "wo", "wg", "w_dkv", "w_ukv", "w_gate", "w_up", "w_down")


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))                 # B = 0 -> identity multiplier at init
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


NF4_CODE = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
            -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
            0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
            0.7229568362236023, 1.0]


def nf4_quantize(W, block=64):
    code = torch.tensor(NF4_CODE, dtype=torch.float32)
    w = W.detach().float().reshape(-1)
    pad = (-w.numel()) % block
    if pad:
        w = F.pad(w, (0, pad))
    w = w.view(-1, block)
    absmax = w.abs().amax(1).clamp(min=1e-8)
    idx = torch.bucketize((w / absmax[:, None]).contiguous(), (code[1:] + code[:-1]) / 2).view(-1).to(torch.uint8)
    if idx.numel() % 2:
        idx = torch.cat([idx, idx.new_zeros(1)])
    packed = (idx[0::2] << 4) | idx[1::2]
    return packed.contiguous(), absmax, code


def nf4_dequantize(packed, absmax, code, shape, block):
    idx = torch.stack([(packed >> 4).long(), (packed & 15).long()], 1).view(-1)
    n = 1
    for s in shape:
        n *= s
    w = code[idx][:absmax.numel() * block].view(-1, block) * absmax[:, None]
    return w.view(-1)[:n].view(*shape)


class QLoRALinear(nn.Module):
    def __init__(self, base, r, alpha, block=64, dropout=0.05):
        super().__init__()
        self.in_features, self.out_features = base.in_features, base.out_features
        self.shape, self.block, self.scale = tuple(base.weight.shape), block, alpha / r
        packed, absmax, code = nf4_quantize(base.weight, block)
        self.register_buffer("packed", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("code", code)
        with torch.no_grad():
            W = base.weight.detach().float()
            self.rel_err = float((nf4_dequantize(packed, absmax, code, self.shape, block) - W).norm() / W.norm().clamp(min=1e-8))
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        W = nf4_dequantize(self.packed, self.absmax, self.code, self.shape, self.block).to(x.dtype)
        return F.linear(x, W) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class IA3Out(nn.Module):
    """y = W0 x * l (keys / values)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.in_features, self.out_features = base.in_features, base.out_features
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l.to(x.dtype)


class IA3In(nn.Module):
    """y = W0 (x * l) (FFN hidden activations, i.e. the input of the down-projection)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.in_features, self.out_features = base.in_features, base.out_features
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))


def apply(method, bb, cfg):
    """The caller has already set requires_grad=False on every backbone parameter."""
    info = {"method": method}
    nL = len(blocks_of(bb))
    if method in ("full", "kd"):
        for p in bb.parameters():
            p.requires_grad = True
        n = sum(1 for _ in bb.parameters())
    elif method == "partial":
        top = cfg["n_top"]
        for blk in list(blocks_of(bb))[nL - top:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in final_norm_of(bb).parameters():
            p.requires_grad = True
        info["train_from"] = nL - top
        n = top
    elif method == "qlora":
        before = sum(p.numel() * p.element_size() for p in bb.parameters())
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: QLoRALinear(l, cfg["r"], cfg["alpha"], cfg.get("block", 64)))
        q = [m for m in bb.modules() if isinstance(m, QLoRALinear)]
        info["nf4_rel_error"] = float(sum(m.rel_err for m in q) / max(1, len(q)))
        info["base_linear_bytes_nf4"] = int(sum(m.packed.numel() + m.absmax.numel() * 4 for m in q))
        info["backbone_bytes_fp32_before"] = int(before)
        print(f"[qlora] {len(q)} linears in NF4: {info['base_linear_bytes_nf4'] / 1e6:.1f} MB (fp32 backbone {before / 1e6:.1f} MB) | "
              f"mean relative weight error {info['nf4_rel_error']:.4f}")
    elif method == "lorma":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: LoRMALinear(l, cfg["r"], cfg["alpha"]))
    elif method == "ia3":
        n_out = replace_linears(bb, ("wk", "wv", "w_ukv"), IA3Out)
        n_in = replace_linears(bb, ("w_down",), IA3In)
        info.update(n_out=n_out, n_in=n_in)
        n = n_out + n_in
        print(f"[ia3] scaling vectors on {n_out} key/value projections and {n_in} FFN down-projection inputs")
    elif method == "prompt":
        emb = bb.tok_emb.weight
        g = torch.Generator().manual_seed(0)
        ids = torch.randint(100, emb.shape[0], (cfg["n_prompt"],), generator=g)
        bb.prompt = nn.Parameter(emb.detach()[ids].clone().float())
        n = cfg["n_prompt"]
    else:
        raise ValueError(f"unknown method {method}")
    assert n > 0, f"{method}: nothing to adapt"
    info["n_modules"] = n
    print(f"[{method}] applied ({n} modules)", {k: v for k, v in cfg.items() if k != "targets"})
    return info


Writing pcr_peft.py


In [6]:
%%writefile pcr_run.py
"""Phase 7 PCR - one experiment. Usage: python pcr_run.py <method> <arch> [tag]
method: full | partial | qlora | lorma | ia3 | prompt | kd      arch: gqa | mla | retnet | encdec
Needs pcr_cache/data.pkl from pcr_prep.py. Env: PCR_SMOKE=1 (tiny pipeline check),
PCR_OVERRIDES='{"NEG_MODE": "random"}' (override any upper-case setting below)."""
import os, sys, json, math, time, random, glob, pickle, hashlib
from pathlib import Path
import numpy as np
import torch

METHOD, ARCH = sys.argv[1], sys.argv[2]
TAG = sys.argv[3] if len(sys.argv) > 3 else ""
SEED = 42

# ---- document input
CHUNK = 512
STORE = 16            # chunks kept per document, evenly spaced over the whole judgment (first and last always)
K_TRAIN = 3           # random stored chunks per document in a training step
K_EVAL = 4            # evenly spaced chunks per document for the dev checks during training
K_INFER = 8           # evenly spaced chunks per document for the final dev / test scores (8 x 512 = 4096 tokens)
K_KD = 4              # chunks per document when scoring the training lists (the KD teacher signal)
DIM = 256
# ---- optimisation
BS, N_NEG = 6, 2      # queries per step; negatives per query (+ the other queries' documents as in-batch negatives)
NEG_MODE = "bm25"     # bm25: hard negatives from the BM25 top-60 | random: uniform pool documents (ablation)
EPOCHS = 8
HEAD_LR, WD, WARMUP, GRAD_CLIP = 1e-3, 0.01, 0.06, 1.0
MAX_MIN = 40          # training time budget (minutes); the best dev-MAP checkpoint is kept
EVAL_EVERY = 120      # steps between dev checks
EVAL_BS = 48

MCFG_ALL = {
    "full":    dict(lr=2e-5),
    "partial": dict(lr=5e-5, n_top=6),
    "qlora":   dict(lr=3e-4, r=16, alpha=32, block=64),
    "lorma":   dict(lr=5e-4, r=16, alpha=16),
    "ia3":     dict(lr=3e-3),
    "prompt":  dict(lr=3e-3, n_prompt=20),
    "kd":      dict(lr=3e-5, keep=None, alpha=0.5, T=0.5, n_neg=7, bs=4, n_teachers=3),
}
for _k, _v in json.loads(os.environ.get("PCR_OVERRIDES", "{}")).items():
    globals()[_k] = _v
SMOKE = os.environ.get("PCR_SMOKE") == "1"
if SMOKE:
    EPOCHS, MAX_MIN, EVAL_EVERY, K_INFER, K_EVAL, K_KD, STORE = 1, 3, 6, 2, 2, 2, 4
MCFG = dict(MCFG_ALL[METHOD])
if METHOD == "kd":
    N_NEG, BS = MCFG["n_neg"], MCFG["bs"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK = ROOT / f"pcr_{METHOD}_{ARCH}{TAG}{'_smoke' if SMOKE else ''}"
WORK.mkdir(parents=True, exist_ok=True)
print("device:", device, "| out:", WORK, "| cfg:", MCFG, "| negatives:", NEG_MODE, flush=True)

import pcr_lib, pcr_peft
from pcr_metrics import (gold_matrix, full_report, fmt, zrow, tune, honest_tune, fuse, write_prediction, prf_at_k, topk)
from tokenizers import Tokenizer
amp, AMP_DTYPE = pcr_lib.get_amp(device)

# ------------------------------------------------------------------ data
blob = pickle.load(open(ROOT / "pcr_cache" / "data.pkl", "rb"))
DATA, META = blob["data"], blob["meta"]
tr, dv, ts = DATA["train"], DATA["dev"], DATA["test"]
DEV_Q = list(range(len(dv["qids"])))
TEST_Q = list(range(len(ts["qids"])))
TRAIN_Q = list(range(len(tr["qids"])))
if SMOKE:
    DEV_Q, TEST_Q, TRAIN_Q = DEV_Q[:24], TEST_Q[:6], TRAIN_Q[:60]
G_dev, NG_dev = dv["G"][DEV_Q], dv["ng"][DEV_Q]

export_dir = pcr_lib.find_p1_export(ARCH)
assert export_dir is not None, f"Phase-1 export for {ARCH} not found under /kaggle/input"
tok = Tokenizer.from_file(str(export_dir / "tokenizer.json"))
TOKH = pcr_lib.file_hash(export_dir / "tokenizer.json")
t0 = time.time()
TOK = {}
for s in ("train", "dev", "test"):
    for part, key in (("q", "qtext"), ("c", "ctext")):
        cp = ROOT / "pcr_cache" / f"tok_{TOKH}_{CHUNK}_{STORE}_{s}_{part}.pkl"
        TOK[s, part] = pcr_lib.tokenize_docs(DATA[s][key], tok, CHUNK, STORE, cp)
print(f"tokenised in {time.time() - t0:.0f}s (cached per tokenizer)")
for s in ("train", "dev", "test"):
    for part in ("q", "c"):
        n = TOK[s, part][1]
        print(f"  {s} {'queries' if part == 'q' else 'pool'}: tokens median {np.median(n):.0f} p90 {np.percentile(n, 90):.0f} | "
              f"> {K_INFER * CHUNK} tokens (more than the final model reads, evenly spaced chunks are used): {np.mean(n > K_INFER * CHUNK):.2f}")

# training structure (pool indices)
gold_in = [list(np.where(tr["G"][i])[0]) for i in TRAIN_Q]
S = dict(qdocs=[TOK["train", "q"][0][i] for i in TRAIN_Q], cdocs=TOK["train", "c"][0], gold=gold_in,
         gold_set=[set(g) for g in gold_in], lists=[list(tr["lists"][i]) for i in TRAIN_Q],
         excl=[set(np.where(tr["mask"][i])[0]) for i in TRAIN_Q], n_pool=len(tr["cids"]))

# ------------------------------------------------------------------ model
arch_mod, bcfg, bb = pcr_lib.load_p1(export_dir)
P1_PARAMS = sum(p.numel() for p in bb.parameters())
print(f"Phase-1 {ARCH}: {P1_PARAMS / 1e6:.1f}M params, kind {bb.kind}, positions {pcr_lib.max_positions(bb)}")
TEACHER, TEACHER_INFO = None, None
if METHOD == "kd":
    runs = []
    for f in glob.glob(str(ROOT / "pcr_*/kd_train.pkl")):
        d = Path(f).parent
        if ("_smoke" in d.name) != SMOKE or "_randneg" in d.name or d.name.startswith("pcr_kd") or not (d / "results.json").exists():
            continue
        runs.append((json.load(open(d / "results.json"))["dev_dense"]["map"], d))
    assert runs, "KD needs kd_train.pkl from the earlier runs"
    runs = sorted(runs, reverse=True)[:MCFG["n_teachers"]]
    acc = {}
    for _, d in runs:
        kt = pickle.load(open(d / "kd_train.pkl", "rb"))
        for q, (idx, sc) in kt.items():
            z = (sc - sc.mean()) / max(1e-6, sc.std())
            a = acc.setdefault(q, {})
            for j, v in zip(idx.tolist(), z.tolist()):
                a.setdefault(j, []).append(v)
    full_q = {q: {j: float(np.mean(v)) for j, v in a.items() if len(v) == len(runs)} for q, a in acc.items()}
    pos_of = {qi: k for k, qi in enumerate(TRAIN_Q)}
    TEACHER = [full_q.get(TRAIN_Q[k], {}) for k in range(len(TRAIN_Q))]
    TEACHER_INFO = [dict(run=d.name, dev_map=m) for m, d in runs]
    print("teacher = mean per-query z-scored similarities of", TEACHER_INFO, "| queries covered", sum(1 for t in TEACHER if t))
    nL = len(pcr_lib.blocks_of(bb))
    keep = MCFG["keep"] or list(range(0, nL, 2))
    bb, bcfg = pcr_lib.make_student_bb(arch_mod, bcfg, bb, keep)
    print(f"student keeps layers {keep} of {nL}: {sum(p.numel() for p in bb.parameters()) / 1e6:.1f}M params")

pcr_lib.strip_backbone(bb)
pcr_lib.patch_encoder_padding_mask(bb)          # before any wrapper replaces the attention projections
for p in bb.parameters():
    p.requires_grad = False
INFO = pcr_peft.apply(METHOD, bb, MCFG)
assert CHUNK + MCFG.get("n_prompt", 0) <= pcr_lib.max_positions(bb)
model = pcr_lib.PCRModel(bb, DIM, train_from=INFO.get("train_from", 0), ckpt=True).to(device)
tot, trn = pcr_lib.count_params(model)
print(f"model {tot / 1e6:.1f}M params | trainable {trn / 1e6:.3f}M", flush=True)


def dev_scores(m, k, kind="maxsim", raw=False):
    QV, QM = pcr_lib.encode_docs(m, [TOK["dev", "q"][0][i] for i in DEV_Q], k, device, amp, EVAL_BS, raw=raw)
    CV, CM = pcr_lib.encode_docs(m, TOK["dev", "c"][0], k, device, amp, EVAL_BS, raw=raw)
    out = {}
    for kd_ in (["maxsim", "mean"] if kind == "both" else [kind]):
        Sx = pcr_lib.score_matrix(QV, QM, CV, CM, device, kd_)
        Sx[dv["mask"][DEV_Q]] = -np.inf
        out[kd_] = Sx
    return out


def eval_fn(m):
    Sd = dev_scores(m, K_EVAL)["maxsim"]
    rep = full_report(Sd, G_dev, NG_dev)
    return {k: rep[k] for k in ("map", "f1@7", "R@7", "R@50")}


ZERO = None
if METHOD != "kd":
    t1 = time.time()
    zs = dev_scores(model, K_EVAL, "both", raw=True)
    ZERO = {k: full_report(v, G_dev, NG_dev) for k, v in zs.items()}
    for k, v in ZERO.items():
        print(f"ZERO-SHOT Phase-1 {ARCH} (mean-pooled states, {k}, {K_EVAL} chunks): {fmt(v)} | {time.time() - t1:.0f}s")

TCFG = dict(lr=MCFG["lr"], head_lr=HEAD_LR, wd=WD, warmup=WARMUP, epochs=EPOCHS, bs=BS, n_neg=N_NEG, neg_mode=NEG_MODE,
            k_train=K_TRAIN, grad_clip=GRAD_CLIP, max_min=MAX_MIN, seed=SEED, eval_every=EVAL_EVERY,
            kd_alpha=MCFG.get("alpha", 1.0), kd_T=MCFG.get("T", 1.0))
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t1 = time.time()
hist, best = pcr_lib.train(model, S, TCFG, device, amp, AMP_DTYPE, eval_fn, teacher=TEACHER)
TRAIN_MIN = (time.time() - t1) / 60
PEAK = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0

# ------------------------------------------------------------------ final scores
t1 = time.time()
Sdev = dev_scores(model, K_INFER, "both")
DEV_SEC = time.time() - t1
REP = {k: full_report(v, G_dev, NG_dev) for k, v in Sdev.items()}
for k, v in REP.items():
    print(f"DEV dense ({k}, {K_INFER} chunks): {fmt(v)}")
KIND = max(REP, key=lambda k: REP[k]["map"])
print(f"similarity used: {KIND} (best dev MAP) | dev encoding + scoring {DEV_SEC:.0f}s")
Sd = Sdev[KIND]
Zd, Zb = zrow(Sd), zrow(dv["bm25"][DEV_Q])
YB, POP = dv["ybad"][DEV_Q], dv["pop"]
H = honest_tune(Zd, Zb, YB, POP, G_dev, NG_dev)
PRM, F_T = tune(Zd, Zb, YB, POP, G_dev, NG_dev)
HYB = full_report(fuse(Zd, Zb, YB, POP, *PRM), G_dev, NG_dev)
print(f"DEV hybrid (w_dense, year_pen, pop) = {PRM}: {fmt(HYB)}")
print(f"honest dev F1@7 (weights fitted on one half, scored on the other): hybrid {H['tuned']:.4f} | dense only {H['dense_only']:.4f} "
      f"| BM25 only {H['bm25_only']:.4f}")

# error analysis on dev (hybrid): per-query hits by gold-set size and by query length
T7 = topk(fuse(Zd, Zb, YB, POP, *PRM))
hits = np.take_along_axis(G_dev, T7, 1).sum(1)
qlen = TOK["dev", "q"][1][DEV_Q]
ERR = {}
for name, lo, hi in (("gold 1-3", 1, 3), ("gold 4-7", 4, 7), ("gold 8+", 8, 10 ** 9)):
    m = (NG_dev >= lo) & (NG_dev <= hi)
    if m.any():
        ERR[name] = dict(n=int(m.sum()), f1=prf_at_k(fuse(Zd, Zb, YB, POP, *PRM), G_dev, NG_dev, rows=np.where(m)[0])[0], zero_hit=float(np.mean(hits[m] == 0)))
qt = np.quantile(qlen, [1 / 3, 2 / 3])
for name, m in (("short queries", qlen <= qt[0]), ("medium queries", (qlen > qt[0]) & (qlen <= qt[1])), ("long queries", qlen > qt[1])):
    if m.any():
        ERR[name] = dict(n=int(m.sum()), f1=prf_at_k(fuse(Zd, Zb, YB, POP, *PRM), G_dev, NG_dev, rows=np.where(m)[0])[0], zero_hit=float(np.mean(hits[m] == 0)))
print("DEV error analysis (hybrid):", {k: {kk: round(vv, 4) if isinstance(vv, float) else vv for kk, vv in v.items()} for k, v in ERR.items()})

# test
t1 = time.time()
QV, QM = pcr_lib.encode_docs(model, [TOK["test", "q"][0][i] for i in TEST_Q], K_INFER, device, amp, EVAL_BS)
CV, CM = pcr_lib.encode_docs(model, TOK["test", "c"][0], K_INFER, device, amp, EVAL_BS)
St = pcr_lib.score_matrix(QV, QM, CV, CM, device, KIND)
St[ts["mask"][TEST_Q]] = -np.inf
TEST_SEC = time.time() - t1
qid_dev = [dv["qids"][i] for i in DEV_Q]
qid_test = [ts["qids"][i] for i in TEST_Q]
write_prediction(qid_dev, dv["cids"], fuse(Zd, Zb, YB, POP, *PRM), WORK / "dev_prediction.json", None)
write_prediction(qid_dev, dv["cids"], Zd, WORK / "dev_prediction_dense.json", None)
Zt = zrow(St)
write_prediction(qid_test, ts["cids"], fuse(Zt, zrow(ts["bm25"][TEST_Q]), ts["ybad"][TEST_Q], ts["pop"], *PRM), WORK / "prediction.json", WORK / "submission.zip")
write_prediction(qid_test, ts["cids"], Zt, WORK / "prediction_dense.json", WORK / "submission_dense.zip")
np.save(WORK / "scores_dev.npy", Sd.astype(np.float32))
np.save(WORK / "scores_test.npy", St.astype(np.float32))
json.dump(dict(dev_q=qid_dev, test_q=qid_test, kind=KIND), open(WORK / "score_ids.json", "w"))
print(f"test: {len(TEST_Q)} queries x {len(ts['cids'])} candidates scored in {TEST_SEC:.0f}s" + (" | SMOKE: not a valid submission" if SMOKE else ""))

# KD teacher signal: similarities on every training query's list (gold + BM25 top-60)
KD_SEC = 0.0
if METHOD != "kd" and NEG_MODE == "bm25":
    t1 = time.time()
    need = sorted({j for l in S["lists"] for j in l})
    QV, QM = pcr_lib.encode_docs(model, S["qdocs"], K_KD, device, amp, EVAL_BS)
    CV, CM = pcr_lib.encode_docs(model, [S["cdocs"][j] for j in need], K_KD, device, amp, EVAL_BS)
    col = {j: c for c, j in enumerate(need)}
    kd_out = {}
    with torch.no_grad():
        for k, qi in enumerate(TRAIN_Q):
            idx = np.asarray(S["lists"][k])
            cv = CV[[col[j] for j in idx]].to(device).float()
            cm = CM[[col[j] for j in idx]].to(device)
            s = pcr_lib.maxsim(QV[k:k + 1].to(device).float(), QM[k:k + 1].to(device), cv, cm)[0].float().cpu().numpy()
            kd_out[qi] = (idx.astype(np.int32), s.astype(np.float32))
    pickle.dump(kd_out, open(WORK / "kd_train.pkl", "wb"), protocol=4)
    KD_SEC = time.time() - t1
    print(f"KD signal written for {len(kd_out)} training queries ({len(need)} candidates) in {KD_SEC:.0f}s")

res = dict(run=WORK.name, method=METHOD, arch=ARCH, mcfg=MCFG, info=INFO, neg_mode=NEG_MODE, params_total=tot, params_trainable=trn,
           phase1_params=P1_PARAMS, train_min=TRAIN_MIN, peak_gpu_gb=PEAK, best_dev_map_during_training=best, history=hist,
           zero_shot=ZERO, dev_dense=REP[KIND], dev_dense_both=REP, kind=KIND, dev_hybrid=HYB, fusion=PRM, honest=H, error_analysis=ERR,
           dev_seconds=DEV_SEC, test_seconds=TEST_SEC, kd_signal_seconds=KD_SEC, k_infer=K_INFER, chunk=CHUNK, dim=DIM, teacher=TEACHER_INFO,
           n_train_queries=len(TRAIN_Q), n_dev=len(DEV_Q), n_test=len(TEST_Q))
json.dump(res, open(WORK / "results.json", "w"), indent=1, default=float)
print(f"\nSUMMARY {WORK.name}: dev dense F1@7 {REP[KIND]['f1@7']:.4f} MAP {REP[KIND]['map']:.4f} R@50 {REP[KIND]['R@50']:.4f} | "
      f"hybrid F1@7 {HYB['f1@7']:.4f} (honest {H['tuned']:.4f}, BM25 alone {H['bm25_only']:.4f}) | params {tot / 1e6:.1f}M "
      f"trainable {trn / 1e6:.2f}M | train {TRAIN_MIN:.0f} min | peak {PEAK:.1f} GB | test scoring {TEST_SEC:.0f}s")
if METHOD == "kd" and TEACHER_INFO:
    print(f"KD: student dev MAP {REP[KIND]['map']:.4f} vs best teacher {max(t['dev_map'] for t in TEACHER_INFO):.4f} | student "
          f"{tot / 1e6:.1f}M params")
print("RUN_DONE", WORK.name, flush=True)


Writing pcr_run.py


In [7]:
%%writefile pcr_ens.py
"""Phase 7 - ensemble of finished runs: average of the runs' per-query z-scored dense similarities, fused with BM25
(+ year penalty + popularity) with weights tuned on dev. Members are chosen by greedy forward selection on the honest
dev F1@7 (fusion weights fitted on one half of the dev queries, scored on the other).
Writes submissions/ensemble_greedy.zip and submissions/ensemble_all.zip. Usage: python pcr_ens.py [--smoke]"""
import json, glob, sys, pickle
from pathlib import Path
import numpy as np
sys.path.insert(0, ".")
from pcr_metrics import zrow, tune, honest_tune, fuse, full_report, fmt, write_prediction

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUB = ROOT / "submissions"; SUB.mkdir(exist_ok=True)
SMOKE = "--smoke" in sys.argv
DATA = pickle.load(open(ROOT / "pcr_cache" / "data.pkl", "rb"))["data"]
dv, ts = DATA["dev"], DATA["test"]

runs = {}
for f in sorted(glob.glob(str(ROOT / "pcr_*/scores_test.npy"))):
    d = Path(f).parent
    if ("_smoke" in d.name) != SMOKE or not (d / "results.json").exists():
        continue
    ids = json.load(open(d / "score_ids.json"))
    runs[d.name] = dict(dev=np.load(d / "scores_dev.npy"), test=np.load(d / "scores_test.npy"), ids=ids, res=json.load(open(d / "results.json")))
names = sorted(runs)
assert names, "no finished runs"
ids0 = runs[names[0]]["ids"]
names = [n for n in names if runs[n]["ids"]["dev_q"] == ids0["dev_q"] and runs[n]["ids"]["test_q"] == ids0["test_q"]]
qpos = {q: i for i, q in enumerate(dv["qids"])}
DQ = [qpos[q] for q in ids0["dev_q"]]
tpos = {q: i for i, q in enumerate(ts["qids"])}
TQ = [tpos[q] for q in ids0["test_q"]]
G, NG = dv["G"][DQ], dv["ng"][DQ]
Zb, YB, POP = zrow(dv["bm25"][DQ]), dv["ybad"][DQ], dv["pop"]
Zbt, YBt, POPt = zrow(ts["bm25"][TQ]), ts["ybad"][TQ], ts["pop"]
Z = {n: zrow(runs[n]["dev"]) for n in names}
ZT = {n: zrow(runs[n]["test"]) for n in names}

print(f"{len(names)} runs | dev queries {len(DQ)} | test queries {len(TQ)}\n")
print(f"{'run':>30} | {'dense F1@7':>10} {'MAP':>7} {'R@50':>7} | {'hybrid F1@7':>11} {'honest':>7} | {'trainable':>10} {'train min':>9} {'peak GB':>7}")
score = {}
for n in names:
    r = runs[n]["res"]
    score[n] = r["honest"]["tuned"]
    print(f"{n:>30} | {r['dev_dense']['f1@7']:>10.4f} {r['dev_dense']['map']:>7.4f} {r['dev_dense']['R@50']:>7.4f} | "
          f"{r['dev_hybrid']['f1@7']:>11.4f} {r['honest']['tuned']:>7.4f} | {r['params_trainable'] / 1e6:>9.2f}M {r['train_min']:>9.0f} "
          f"{r['peak_gpu_gb']:>7.1f}")
for n in names:
    zs = runs[n]["res"].get("zero_shot")
    if zs:
        print(f"zero-shot Phase-1 {runs[n]['res']['arch']:>6} ({n}): " + " | ".join(f"{k}: F1@7 {v['f1@7']:.4f} MAP {v['map']:.4f}" for k, v in zs.items()))


def mean_z(mem, test=False):
    src = ZT if test else Z
    return zrow(np.mean([src[m] for m in mem], 0))


def evaluate(mem):
    zd = mean_z(mem)
    return honest_tune(zd, Zb, YB, POP, G, NG, reps=2)["tuned"]


pool = [n for n in names if "_randneg" not in n]
members, best = [], -1.0
while len(members) < 6:
    cand = [(evaluate(members + [n]), n) for n in pool if n not in members]
    if not cand:
        break
    f, n = max(cand)
    if f <= best + 1e-4:
        break
    members.append(n); best = f
    print(f"+ {n:>30} -> honest dev F1@7 {f:.4f}")


def build(mem, name):
    zd = mean_z(mem)
    prm, f = tune(zd, Zb, YB, POP, G, NG)
    h = honest_tune(zd, Zb, YB, POP, G, NG)
    rep = full_report(fuse(zd, Zb, YB, POP, *prm), G, NG)
    zname = ("smoke_" if SMOKE else "") + name
    write_prediction(ids0["dev_q"], dv["cids"], fuse(zd, Zb, YB, POP, *prm), ROOT / f"dev_prediction_{zname}.json", None)
    write_prediction(ids0["test_q"], ts["cids"], fuse(mean_z(mem, True), Zbt, YBt, POPt, *prm), ROOT / f"prediction_{zname}.json",
                     SUB / f"{zname}.zip")
    print(f"{zname.upper()} {mem}\n   weights {prm} | dev {fmt(rep)} | honest {h}")
    return dict(members=mem, params=prm, dev=rep, honest=h)


out = {}
if members:
    out["greedy"] = build(members, "ensemble_greedy")
if len(pool) > 1 and sorted(pool) != sorted(members):
    out["all"] = build(pool, "ensemble_all")
json.dump(out, open(ROOT / "ensemble_results.json", "w"), indent=1, default=float)
print("ENS_DONE")


Writing pcr_ens.py


In [8]:
%%writefile evaluate_phase7.py
#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc


def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)


def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")
    p.add_argument("--prediction",type=Path,required=True)
    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")
    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")
    return p.parse_args()


def main():
    a=parse_args()
    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)
    errs=validate(pred,gold,candidates)
    if errs:
        print("LOCAL VALIDATION FAILED")
        for i,e in enumerate(errs,1): print(f"  {i}. {e}")
        print(f"Total validation errors: {len(errs)}")
        raise SystemExit(1)
    result=score(gold,pred)
    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}
    print("Validation passed.")
    print(f"Micro-F1@7: {scores['primary_metric_1']:.10f}")
    print(f"Micro-Precision@7: {scores['secondary_metric_1']:.10f}")
    print(f"Micro-Recall@7: {scores['secondary_metric_2']:.10f}")
    print(f"TP={result['tp']} FP={result['fp']} FN={result['fn']}")
    if a.output:
        a.output.parent.mkdir(parents=True,exist_ok=True)
        with a.output.open('w',encoding='utf-8') as f: json.dump(scores,f,indent=2); f.write('\n')
        print(f"Wrote {a.output}")

if __name__=='__main__': main()


Writing evaluate_phase7.py


In [9]:
# ===== Phase 7 driver: (1) data prep + BM25 on CPU, (2) the six adaptation runs as separate processes, one queue per
# T4, (3) KD (needs the earlier runs' scores) + the negative-sampling ablation, (4) the ensemble and the official
# evaluate_phase7.py on every run's dev predictions. Set SMOKE = True for a ~25-minute pipeline check first.
import os, sys, subprocess, time, glob, shutil, json, py_compile
from pathlib import Path
import torch
SMOKE = False       # True = tiny pipeline check (passed on 4 Nov). False = the real runs (~4 h on 2 x T4).
for f in ("pcr_metrics.py", "pcr_prep.py", "pcr_lib.py", "pcr_peft.py", "pcr_run.py", "pcr_ens.py", "evaluate_phase7.py"):
    py_compile.compile(f, doraise=True)
os.environ["HF_TOKEN"] = HF_TOKEN or ""
W = Path("/kaggle/working")
LOG = W / "logs"; LOG.mkdir(parents=True, exist_ok=True)
T0 = time.time()

print("=== data preparation + BM25 (CPU) ===", flush=True)
r = subprocess.run([sys.executable, "pcr_prep.py"] + (["--smoke"] if SMOKE else []), capture_output=True, text=True)
open(LOG / "prep.log", "w").write(r.stdout + r.stderr)
print(r.stdout[-25000:], r.stderr[-4000:] if r.returncode else "")
assert r.returncode == 0 and (W / "pcr_cache" / "data.pkl").exists(), "prep failed - see above"

NGPU = max(1, torch.cuda.device_count())
# (method, arch, tag, overrides)
QUEUES = {0: [("lorma", "encdec", "", {}), ("full", "gqa", "", {}), ("ia3", "mla", "", {})],
          1: [("qlora", "retnet", "", {}), ("prompt", "gqa", "", {}), ("partial", "mla", "", {})]}
FINAL = {0: [("kd", "gqa", "", {})],                                         # student = 8 of the GQA layers
         1: [("lorma", "encdec", "_randneg", {"NEG_MODE": "random"})]}       # ablation: random instead of BM25 negatives
if NGPU == 1:
    QUEUES = {0: QUEUES[0] + QUEUES[1]}
    FINAL = {0: FINAL[0] + FINAL[1]}


def tail(path, n=1):
    try:
        return open(path).read().strip().splitlines()[-n:]
    except Exception:
        return []


def run_queues(queues):
    pend = {g: list(q) for g, q in queues.items()}
    cur, status, last = {g: None for g in queues}, {}, 0
    while True:
        for g in queues:
            if cur[g] is not None and cur[g][0].poll() is not None:
                p, f, name = cur[g]; f.close(); status[name] = p.returncode
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} finished {name} rc={p.returncode}\n    " + "\n    ".join(tail(LOG / f"{name}.log", 4)), flush=True)
                cur[g] = None
            if cur[g] is None and pend[g]:
                m, a, tag, ov = pend[g].pop(0); name = f"{m}_{a}{tag}"
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PCR_SMOKE="1" if SMOKE else "0", PYTHONUNBUFFERED="1",
                           PCR_OVERRIDES=json.dumps(ov), TOKENIZERS_PARALLELISM="false")
                f = open(LOG / f"{name}.log", "w")
                cur[g] = (subprocess.Popen([sys.executable, "pcr_run.py", m, a, tag], stdout=f, stderr=subprocess.STDOUT, env=env), f, name)
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} started {name}", flush=True)
        if all(c is None for c in cur.values()):
            return status
        if time.time() - last > 300:
            last = time.time()
            for g, c in cur.items():
                if c is not None:
                    print(f"[{(time.time() - T0) / 60:6.1f} min]   GPU{g} {c[2]}: {tail(LOG / (c[2] + '.log'))}", flush=True)
        time.sleep(15)


print("GPUs:", NGPU, "| queues:", QUEUES, "| final:", FINAL, "| smoke:", SMOKE, flush=True)
status = run_queues(QUEUES)
status.update(run_queues(FINAL))
print("\nreturn codes:", status)
sub = W / "submissions"; sub.mkdir(exist_ok=True)
for d in glob.glob(str(W / "pcr_*")):
    d = Path(d)
    if (d / "submission.zip").exists():
        shutil.copy(d / "submission.zip", sub / f"{d.name}_hybrid.zip")
    if (d / "submission_dense.zip").exists():
        shutil.copy(d / "submission_dense.zip", sub / f"{d.name}_dense.zip")
r = subprocess.run([sys.executable, "pcr_ens.py"] + (["--smoke"] if SMOKE else []), capture_output=True, text=True)
print(r.stdout[-20000:], r.stderr[-5000:] if r.returncode else "")

print("\n=== official evaluate_phase7.py on the dev predictions ===")
truth = json.load(open(W / "dev_truth.json"))
preds = sorted(glob.glob(str(W / "pcr_*/dev_prediction*.json"))) + sorted(glob.glob(str(W / "dev_prediction_*.json")))
for p in preds:
    p = Path(p)
    if ("_smoke" in str(p)) != SMOKE and "bm25" not in p.name:
        continue
    pred = json.load(open(p))
    t = W / "dev_truth_subset.json"
    json.dump({q: truth[q] for q in pred}, open(t, "w"))           # the smoke runs score only some dev queries
    r = subprocess.run([sys.executable, "evaluate_phase7.py", "--truth", str(t), "--prediction", str(p), "--candidate-ids",
                        str(W / "candidate_ids_dev.json")], capture_output=True, text=True)
    lines = [l for l in r.stdout.split("\n") if l.startswith(("Micro-F1", "LOCAL VALIDATION", "  "))]
    print(f"{p.parent.name + '/' if p.parent != W else ''}{p.name} ({len(pred)} queries) | " + " | ".join(lines) + (r.stderr[-300:] if r.returncode else ""))
print(sorted(os.listdir(sub)))
print("\n=== SUMMARIES ===")
for lg in sorted(LOG.glob("*.log")):
    if lg.stem == "prep":
        continue
    for line in open(lg).read().splitlines():
        if line.startswith(("SUMMARY", "KD:", "DEV ", "ZERO-SHOT", "honest", "similarity used", "teacher", "[qlora]", "[ia3]",
                            "Phase-1 ", "restored", "model ", "KD signal", "test:")) or "Traceback" in line or "Error" in line:
            print(lg.stem, "|", line)
print(f"ALL_DONE in {(time.time() - T0) / 60:.0f} min")


=== data preparation + BM25 (CPU) ===
Phase-7 files: ['Phase_7_PCR/Dev/dev.json', 'Phase_7_PCR/Dev/dev_candidates.json', 'Phase_7_PCR/Test/test.json', 'Phase_7_PCR/Test/test_candidates.json', 'Phase_7_PCR/Train/train.json', 'Phase_7_PCR/Train/train_candidates.json']

train: 827 queries | raw dict, example keys ['text', 'relevant_candidates'] | pool 4320 docs, raw dict, example ['text']
  query '0000000599': 'PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 1545 1964 SCR (3) 573 CITATOR INFO : R 1970 SC 765 ('
  candidate '0000000262': 'JUDGMENT <NAME>, C.J. 1. The only point which has to be considered in this appeal is whether or not the lower appellate Court was wrong in admitting in evidence statements contained in two Exhibits, being Ex. 58 and 59 in the suit. 2. The case of the plaintif